# News Recommendation System - Part 1

## Introduction
This notebook performs data analysis and preprocessing for a news recommendation scenario.
The objective is to understand user interactions and prepare features for modeling.

## Dataset Overview
The dataset includes click logs and article metadata/embeddings.
Expected files under `../data/data_raw/`:
- train_click_log.csv
- testA_click_log.csv
- testB_click_log.csv
- articles.csv
- articles_emb.csv


## Guide package

In [ ]:
# %matplotlib inline # Display plot results in Jupyter Notebook

import pandas as pd  # Import the pandas library for data processing and analysis
import numpy as np  # Import the NumPy library for numerical calculations

import matplotlib.pyplot as plt  # Import the matplotlib library for data visualization
import seaborn as sns  # Import the seaborn library for more beautiful data visualization
plt.rc('font', family='SimHei', size=13)  # Set the font to SimHei (Chinese bold) and the font size to 13

import os  # Import the os module for operating system-related functions
import gc  # Import the gc module for garbage collection
import re  # Import the re module for regular expression operations
import warnings  # Import the warnings module to ignore warning messages
import sys  # Import the sys module to access system-related functions

warnings.filterwarnings("ignore")  # Ignore warning messages


## Read data

In [ ]:

# Connect to Google Drive (data transferred to your own Google Drive)


In [ ]:
data_path = '../data/data_raw/'
save_path = '../data/temp_results/'



#####train
trn_click = pd.read_csv(data_path+'train_click_log.csv')
item_df = pd.read_csv(data_path+'articles.csv')
item_df = item_df.rename(columns={'article_id': 'click_article_id'})  # Rename to facilitate subsequent matching
item_emb_df = pd.read_csv(data_path+'articles_emb.csv')

#####test
tst_click = pd.read_csv(data_path+'testA_click_log.csv')

## Data preprocessing
Calculate user click rank and number of clicks

In [ ]:
trn_click.head()

In [ ]:
# Sort click timestamps for each user
trn_click['rank'] = trn_click.groupby(['user_id'])['click_timestamp'].rank(ascending=False).astype(int)
tst_click['rank'] = tst_click.groupby(['user_id'])['click_timestamp'].rank(ascending=False).astype(int)

In [ ]:
trn_click[trn_click['user_id'] == 199999]

In [ ]:
# Count the number of times users click on the article and add a new column count
trn_click['click_cnts'] = trn_click.groupby(['user_id'])['click_timestamp'].transform('count')
tst_click['click_cnts'] = tst_click.groupby(['user_id'])['click_timestamp'].transform('count')

## Data browsing

### User click log file_training set

In [ ]:
trn_click = trn_click.merge(item_df, how='left', on=['click_article_d'])
trn_click.head()

The meaning of each field in the train_click_log.csv file data

1. user_id: unique identifier of the user
2. click_article_id: the unique identifier of the article clicked by the user
3. click_timestamp: the timestamp when the user clicks on the article
4. click_environment: the environment in which the user clicks on the article
5. click_deviceGroup: The device group where the user clicked the article
6. click_os: the operating system when the user clicks on the article
7. click_country: the country where the user clicked on the article
8. click_region: The area where the user clicks on the article
9. click_referrer_type: When the user clicks on the article, the source of the article

In [ ]:
# User click log information
trn_click.info()

In [ ]:
# The number of users in the training set is 200,000
trn_click.user_id.nunique()

In [ ]:
trn_click.groupby('user_id')['click_article_id'].count().min()  # Each user in the training set clicked on at least two articles.


Draw a histogram to roughly look at the basic attribute distribution.

In [ ]:
# Change the Font to Chinese because there is Chinese Font in matplotlib

!wget -q -O TaipeiSansTCBeta-Regular.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.font_manager import fontManager

fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
mpl.rc('font', family='Taipei Sans TC Beta')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_bar_chart(data, column, ax):
    """
    
    """
    value_counts = data[column].value_counts().reset_index()[:10]
    sns.barplot(x=value_counts.index, y=value_counts[column], ax=ax)
    for item in ax.get_xticklabels():
        item.set_rotation(0)
    ax.set_title(column)
    ax.set_xlabel('')

# Create a new graphics window
fig, axes = plt.subplots(5, 2, figsize=(10, 12))

# Place subplots in a 5x2 grid
for i, column in enumerate(['click_article_id', 'click_timestamp', 'click_environment', 'click_deviceGroup',
                            'click_os', 'click_country', 'click_region', 'click_referrer_type', 'rank', 'click_cnts']):
    plot_bar_chart(trn_click, column, axes[i//2, i%2])

plt.tight_layout()
plt.show()


In [ ]:
trn_click['click_deviceGroup'].value_counts(normalize = True)

Judging from the click device group click_deviceGroup, device 1 accounts for the majority (61%), and device 3 accounts for 36%.

### Test set user click log

In [ ]:
tst_click = tst_click.merge(item_df, how='left', on=['click_article_id'])
tst_click.head()

In [ ]:
set(trn_click.user_id) & set(tst_click.user_id)

The users of the training set and the test set are completely different (no intersection)

The user IDs of the training set range from 0 to 199999, while the user IDs of the test set A range from 200000 to 249999.

Therefore, when we train, we need to include the data of the test set, which is called full data.

In [ ]:
# The number of users in the test set is 50,000
tst_click.user_id.nunique()

In [ ]:
tst_click.groupby('user_id')['click_article_id'].count().min() # Note that there are users in the test set who only clicked on the article once.


### News article information data table

In [ ]:
# News article data set browsing
pd.concat([item_df.head(),item_df.tail()])

#### Article word count


1. Overall distribution of word count in articles

See whether the number of words is shorter, or whether long articles are more dominant. See if there are some extreme values, or intervals where the number of words is relatively concentrated. If short articles account for the majority, it may mean that content consumption on the platform is more inclined to browse quickly. If there are many long articles, users may have a certain need for in-depth reading.

In [ ]:
# Plot a histogram of word count distribution
plt.figure(figsize=(10, 6))
sns.histplot(item_df['words_count'], bins=60, kde=True)
plt.title('Word Count Distribution')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# You can also use value_counts for binning
item_df['words_count'].value_counts(bins = 50)[:20].plot(kind = 'barh',rot = 0, figsize = [12,4])

In [ ]:
# More complex charts can also be used

def plot_distribution(data, column):
    """
    :(KDE)、(Boxplot)(Violin Plot)

    :
    - data: DataFrame,
    - column: str,
    """
    plt.figure(figsize=(12, 4))

    # Kernel density estimation plot
    plt.subplot(1, 3, 1)
    sns.kdeplot(data[column], shade=True)
    plt.xlabel(column)
    plt.ylabel('Density')
    plt.title('Density Plot of ' + column)

    # boxplot
    plt.subplot(1, 3, 2)
    sns.boxplot(x=data[column])
    plt.xlabel(column)
    plt.title('Boxplot of ' + column)

    # Violin diagram
    plt.subplot(1, 3, 3)
    sns.violinplot(x=data[column])
    plt.xlabel(column)
    plt.title('Violin Plot of ' + column)

    plt.tight_layout()
    plt.show()

plot_distribution(item_df, 'words_count')

2. The relationship between word count and publishing time

Is there a trend in the word count of articles published in different time periods? If you find that the number of words changes over time, it may indicate that the platform’s content strategy has changed, or that users’ preferences for long and short articles are adjusting.

In [ ]:
# Convert timestamp to date format
item_df['created_at'] = pd.to_datetime(item_df['created_at_ts'], unit='ms')

# Analyze word count trends by year
item_df['year'] = item_df['created_at'].dt.year
plt.figure(figsize=(10, 6))
sns.lineplot(x='year', y='words_count', data=item_df)
plt.title('Word Count Trend Over Time')
plt.xlabel('Year')
plt.ylabel('Average Word Count')
plt.show()

3. The relationship between word count and article type (category_id)

Visually see the word count distribution of articles in different categories and understand whether the word count of certain types of articles is more concentrated or spread out.

In [ ]:
# Group by category and draw a box plot to view word count distribution
plot_distribution(item_df, 'category_id')

#### News article embedding vector representation


The embedding vector representation of news articles refers to representing each news article as a fixed-length vector. This vector is usually obtained by mapping the words or other semantic units in the article to real vectors in a high-dimensional space. The purpose of this representation method is to encode the semantic information of the article into vector form.

Let's say we have the following two news articles:

Article A: "Scientists find new vaccine effective in preventing influenza"
Article B: "National policy is introduced to encourage enterprises to increase investment in environmental protection"
We can use a pretrained word embedding model (such as Word2Vec, GloVe, or FastText) to map each word to a fixed-length vector. Suppose our word embedding model is a 100-dimensional vector.

We can then calculate the embedding vector for each article, there are usually several ways to do this:

Average vector: average the vector of each word in the article to obtain the embedding vector of the entire article.
Weighted average vector: Perform a weighted average of the vectors of each word in the article. The weight can be a TF-IDF value, etc.
Model encoding: Use a deep learning model (such as recurrent neural network, Transformer, etc.) to encode the entire article to obtain a fixed-length vector of the article.
Assuming we use the average vector method, the word vectors in article A are as follows (assuming the vector dimension is 3 for simplicity):

- "Scientist": [0.2, 0.1, 0.5]
- "Discover": [0.3, 0.2, 0.4]
- "New": [0.1, 0.4, 0.3]
- "Vaccine": [0.5, 0.6, 0.2]
... (vectors of other words)

The embedding vector of article A can be obtained by averaging these word vectors

In [ ]:
# !pip uninstall numpy gensim
# !pip install numpy==1.23.5
# !pip install gensim

In [ ]:
import nltk
import numpy as np
from gensim.models import Word2Vec

# Download the data required by NLTK
nltk.download('punkt')
nltk.download('punkt_tab')

# Article content
article = "This week’s White House Report Card capped an eventful week that showed inflation was still hot, a major pollster predicting President Joe Biden’s reelection defeat, and Russian President Vladimir Putin offering an unwelcome election endorsement."

# participle
tokens = nltk.word_tokenize(article)

# Train the Word2Vec model
sentences = [tokens]  # There is only one article here, but in actual applications there can be multiple sentences
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# Build word embedding dictionary
word_embeddings = {}
for word in set(tokens):
    if word in model.wv:
        word_embeddings[word] = model.wv[word]

# Get the embedding vector of each word
word_vectors = [word_embeddings[word] for word in tokens if word in word_embeddings]

# Calculate the embedding vector of the entire article (average)
article_embedding = sum(word_vectors) / len(word_vectors)

print("The embedding vector of the article:", article_embedding)

In [ ]:
item_emb_df.head()

In [ ]:
item_emb_df.shape

## Data analysis

### User clicks repeatedly

In [ ]:
#####merge
user_click_merge = pd.concat([trn_click,tst_click])

In [ ]:
# User clicks repeatedly
user_click_count = user_click_merge.groupby(['user_id', 'click_article_id'])['click_timestamp'].agg({'count'}).reset_index()
user_click_count[:10]

In [ ]:
user_click_count[user_click_count['count']>7]

In [ ]:
# Number of news clicks by users
user_click_count.loc[:,'count'].value_counts()

It can be seen that 1,605,541 (about 99.2%) users have not read the article repeatedly, and only a very small number of users have clicked on an article repeatedly. This can also be made into a separate feature

### Distribution of the number of news clicks by users

In [ ]:
# Generate cumulative click distribution
user_click_item_count = sorted(user_click_merge.groupby('user_id')['click_article_id'].count(), reverse=True)

# Plot a cumulative distribution
plt.figure(figsize=(10, 6))
plt.hist(user_click_item_count, bins=30, edgecolor='black', alpha=0.7)
plt.title('Distribution chart of user clicks')
plt.xlabel('Clicks')
plt.ylabel('number of users')
plt.grid(True)
plt.show()


You can see how active a user is based on the number of times they click on an article.

In [ ]:
user_click_item_count_series = pd.Series(user_click_item_count)

threshold_high = user_click_item_count_series.quantile(0.8)
print(threshold_high)
threshold_low = user_click_item_count_series.quantile(0.2)
print(threshold_low)

user_activity_level = pd.cut(user_click_item_count_series, bins=[0, threshold_low, threshold_high, float('inf')], labels=['low activity', 'medium active', 'Highly active'])


activity_df = pd.DataFrame({'Activity': user_activity_level})

# Draw a bar chart of activity distribution
plt.figure(figsize=(8, 6))
sns.countplot(data=activity_df, x='Activity', palette='Set2', order=['low activity', 'medium active', 'Highly active'])

# Add titles and tags
plt.title('User activity distribution')
plt.xlabel('activity level')
plt.ylabel('number of users')

In [ ]:
all(clicks > 100 for clicks in user_click_item_count[:50])

The top 50 users with the most clicks all have more than 100 clicks. Idea: We can define users with clicks greater than or equal to 100 times as active users. This is a simple processing idea. To judge user activity, a more comprehensive method is to combine the click time. Later we will judge user activity based on the number of clicks and click time.

In [ ]:
print(f"The number of users who clicked less than or equal to twice is: {(user_click_item_count_series <= 2).sum()}")

It can be seen that there are many users who click less than or equal to twice. These users can be considered as inactive users.

### Analysis of news clicks

In [ ]:
user_click_merge

In [ ]:
item_click_count = sorted(user_click_merge.groupby('click_article_id')['click_timestamp'].count(), reverse=True)

Common analysis directions include the distribution of clicks, statistics of popular articles, long tail effects, etc.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(item_click_count, bins=30, edgecolor='black', alpha=0.7)
plt.title('Distribution chart of article clicks')
plt.xlabel('Clicks')
plt.ylabel('Number of articles')
plt.grid(True)
plt.show()

In the data of article clicks, this phenomenon usually appears:

- A small number of articles will have very high clicks (such as popular articles).
- Most articles have fewer clicks, maybe even only 1 to 2 clicks (long tail effect).

In a regular histogram, articles with very high clicks may "compress" articles with low clicks.

The logarithmic distribution makes these widely different data more comparable by logarithmically transforming the number of clicks.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(item_click_count, bins=30, edgecolor='black', alpha=0.7, log=True)
plt.title('Logarithmic distribution chart of article clicks')
plt.xlabel('Clicks')
plt.ylabel('Number of articles (log)')
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(item_click_count[:100], bins=30, edgecolor='black', alpha=0.7)
plt.title('Distribution chart of article clicks')
plt.xlabel('Clicks')
plt.ylabel('Number of articles')
plt.grid(True)
plt.show()

You can see the top 100 news articles with the most clicks, with more than 4,000 clicks.

In [ ]:
# Get the top 10 articles with the most clicks
top_10_articles = item_click_count[:10]

print("The number of clicks on the top 10 articles with the most clicks are:")
for i, count in enumerate(top_10_articles, 1):
    print(f"Number of clicks on article {i}: {count}")


The top 20 news articles with the most clicks have more than 2,500 clicks. Idea: These news can be defined as hot news. This is also a simple processing method. Later, we will also divide the article popularity based on the number of clicks and time.

Check articles with very few clicks and analyze whether there is a significant long tail effect, that is, many articles have very low clicks.

In [ ]:
# Count the number of articles with less than or equal to 2 clicks
low_click_count_articles = sum(clicks <= 2 for clicks in item_click_count)

print(f"The number of articles with less than or equal to 2 clicks is: {low_click_count_articles}")

It can be found that many news items are only clicked once or twice. Idea: These news can be defined as unpopular news

### News co-occurrence frequency: the number of times two news articles appear consecutively


The definition of news co-occurrence frequency is: the number of times a certain news article (click_article_id) and another news article (next_item) appear continuously in the user's click behavior. This analysis can reveal users' reading habits and preferences.

If certain articles are frequently clicked together, it may mean that the two articles are related in topic, content, or time.

In [ ]:
tmp = user_click_merge.sort_values('click_timestamp')
tmp['next_item'] = tmp.groupby(['user_id'])['click_article_id'].transform(lambda x:x.shift(-1))
union_item = tmp.groupby(['click_article_id','next_item'])['click_timestamp'].agg({'count'}).reset_index().sort_values('count', ascending=False)
union_item[['count']].describe()

In [ ]:
# Get the article pairs with the top 20 co-occurrence frequencies
top_20_pairs = union_item.nlargest(20, 'count')

import networkx as nx
import matplotlib.pyplot as plt

# Create diagram
G = nx.from_pandas_edgelist(top_20_pairs, 'click_article_id', 'next_item', ['count'])

# Set drawing parameters
plt.figure(figsize=(6, 6))
pos = nx.spring_layout(G)  # Select layout
nx.draw(G, pos, with_labels=True, node_size=2000, node_color='skyblue', font_size=10, font_color='black', edge_color='gray', alpha=0.7)

# Draw labels for edges showing co-occurrence frequencies
labels = nx.get_edge_attributes(G, 'count')
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels)

# Set title
plt.title('Network diagram of articles with top 10 co-occurrence frequencies')
plt.show()



In [ ]:
pair_detail = item_df[item_df['click_article_id'].isin(set(top_20_pairs['click_article_id'].astype(float).astype(int)) | set(top_20_pairs['next_item'].astype(float).astype(int)))]
pair_detail

### News article information

In [ ]:
# The number of times different types of news appear
user_click_merge['category_id'].value_counts()

### Preference for the type of news clicked by the user

This feature can be used to measure whether the user's interests are broad.

In [ ]:
user_interest_count = user_click_merge.groupby('user_id')['category_id'].nunique()
user_interest_count

In [ ]:
interest_bins = [0, 1, 3, 5, 10, 20, float('inf')]
labels = ['Category 1', 'Category 2-3', 'Category 4-5', 'Category 6-10', 'Category 10-20','Category 20 and above']
interest_categories = pd.cut(user_interest_count, bins=interest_bins, labels=labels, right=False)

plt.figure(figsize=(5, 5))
interest_distribution = interest_categories.value_counts()
plt.pie(interest_distribution, labels=interest_distribution.index, autopct='%1.1f%%', startangle=140)
plt.title('Pie chart of distribution of number of user interest categories')
plt.axis('equal')  # Keep the pie chart circular
plt.show()


It can be seen from the above figure that a small number of users have extremely broad reading types, and most of them are in less than 20 news types.

### Analysis of time when users click on news

In [ ]:
# about six minutes
def mean_diff_time_func(df, col):
    # Convert timestamp to datetime format
    df[col] = pd.to_datetime(df[col], unit='ms')  # Assume the timestamp is in milliseconds

    # Calculate the previous time point
    df['time_shift1'] = df[col].shift(1)

    # Calculate time difference and convert to minutes
    df['diff_time'] = (df[col] - df['time_shift1']).dt.total_seconds() / 60

    return df['diff_time'].mean()


# First sort the data by user and click timestamp
user_click_merge_sorted = user_click_merge.sort_values(by=['user_id', 'click_timestamp'])

# Then filter out users with more than 2 clicks
filtered_users = user_click_merge_sorted.groupby('user_id').filter(lambda x: len(x) > 2)


mean_diff_click_time = filtered_users.groupby('user_id').apply(lambda x: mean_diff_time_func(x, 'click_timestamp'))

In [ ]:
mean_diff_click_time_df = mean_diff_click_time.reset_index()

# Draw a distribution map
plt.figure(figsize=(10, 6))
sns.histplot(mean_diff_click_time_df[0], bins=30, kde=True)  # Overlay kernel density estimation curves using kde=True
plt.title('Average distribution chart of click time difference')
plt.xlabel('Click time difference (minutes)')
plt.xlim(0,12000)
plt.ylabel('number of users')
plt.grid()
plt.show()

From the picture above, we can see that the time difference between different users clicking on the article is different.

## Summary

Through the process of data analysis, we can currently obtain the following important information, which is very helpful for us in subsequent feature creation and analysis:
1. The user IDs in the training set and the test set do not overlap, that is, the user models in the test set have not been seen before.
2. The minimum number of articles clicked by users in the training set is 2, while the minimum number of articles clicked by users in the test set is 1
3. Users have repeated clicks on articles, but these are all present in the training set.
4. The clicking environment of the same user is not unique. Statistical features can be used when doing this part of the features later.
5. The number of times a user clicks on an article has a great degree of distinction. Later, we can use this to create features to measure user activity.
6. The number of times an article is clicked by users also has a great degree of distinction. Later, we can use this to create features to measure the popularity of the article.
7. The news that users watch is highly relevant, so often when we judge whether a user is interested in an article, it will be largely related to the articles he has clicked on in the past.
8. There is a big difference in the number of words in the articles clicked by users. This can reflect the difference in the number of words in the articles that users have.
9. The topics of articles that users have clicked are also very different, which can reflect the user’s topic preferences.
10. The time difference between different users clicking on the article will also be different, which can reflect the user's preference for the timeliness of the article.

Therefore, based on some of the above analysis, it can better help us do feature engineering later and fully mine the hidden information of the data.

## Insights

1. **User uniqueness:** There is no duplication of user IDs in the training set and the test set, and the user model in the test set has never been seen before. This indicates that the model will encounter new users during the testing phase and needs to consider cold start issues, such as how to make recommendations for new users.
2. **User behavioral characteristics:**
   - In the training set, the minimum number of articles clicked by users is 2, while in the test set, the minimum number of articles clicked by users is 1. This shows that there may be more sparse user behaviors in the test set, and attention needs to be paid to the robustness of the model.
   - Users repeatedly click on articles, which is an important user behavior characteristic. You can consider counting the number of user clicks on the same article as a feature.
3. **Diversity of click environments:** The click environments of the same user are not unique. When making features, you can consider introducing statistical features related to the click environment, such as the variance of click times, the diversity of click devices, etc.
4. **Characteristics of clicks:**
   - The number of times a user clicks on an article is highly differentiated, which can be used to measure user activity. Features related to the number of user clicks can be constructed, such as the average number of clicks, distribution of clicks, etc.
   - The number of times an article is clicked by users also has a great degree of differentiation and can be used to measure the popularity of the article. Features related to the number of clicks on an article can be constructed, such as the average number of clicks, distribution of clicks, etc.
5. **Relevance of user historical behavior:** There is a strong correlation between the news that users read and the articles that they have clicked on in the past. You can consider building features related to the relevance of articles clicked by users in history, such as articles with similar themes to the articles recently clicked by users.
6. **Article attribute characteristics:**
   - There is a big difference in the word count of articles that users have clicked on, which can be used to reflect the user's preference for the word count of articles. Features related to article word count can be constructed, such as average article word count, word count distribution, etc.
   - The topics of articles clicked by users are quite different, which can be used to reflect the user's topic preferences. Features related to the article topic can be constructed, such as the distribution of topics that users have clicked on, etc.
7. **Time characteristics:** The time difference between different users clicking on an article is different, reflecting the user's preference for the timeliness of the article. Features related to the time difference between users clicking on articles can be constructed, such as the average time difference, the distribution of time differences, etc.